<a href="https://colab.research.google.com/github/BintangPancahaya/PemMes_2026/blob/main/JS06/JS06_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Tugas 1: Klasifikasi voice.csv (SVM - Linier, Poly, RBF)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load Data
df = pd.read_csv('voice.csv')

# 2. Preprocessing
le = LabelEncoder()
df['label'] = le.fit_transform(df['label']) # male/female -> 1/0

X = df.drop('label', axis=1)
y = df['label']

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Skenario Split dan Kernel
ratios = [0.3, 0.2] # 70:30 dan 80:20
kernels = ['linear', 'poly', 'rbf']
results = []

# 4. Looping Training dan Evaluasi
for ratio in ratios:
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=ratio, random_state=42)

    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        train_ratio = int((1-ratio)*100)
        test_ratio = int(ratio*100)

        results.append({
            'Rasio (Train:Test)': f'{train_ratio}:{test_ratio}',
            'Kernel': kernel,
            'Akurasi': acc
        })

# 5. Tabulasi Hasil
df_hasil = pd.DataFrame(results)
print("--- Hasil Evaluasi voice.csv ---")
print(df_hasil.to_string(index=False))

--- Hasil Evaluasi voice.csv ---
Rasio (Train:Test) Kernel  Akurasi
             70:30 linear 0.970557
             70:30   poly 0.956887
             70:30    rbf 0.981073
             80:20 linear 0.976341
             80:20   poly 0.968454
             80:20    rbf 0.982650


2. Tugas 2: Klasifikasi Citra Siang/Malam (SVM Kernel RBF + Grid Search)

In [3]:
import zipfile

# Ekstrak file images.zip ke direktori saat ini
with zipfile.ZipFile('images.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [4]:
from pathlib import Path

# Sesuaikan nama folder dengan hasil ekstraksi zip
train_dir = './images'  # Atau Path('./images')

In [9]:
	# Image directories
	train_dir = "images/training/"
	test_dir = "images/test/"

In [11]:
import numpy as np
from pathlib import Path
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import glob

# ==========================================
# 0. LOAD DATASET DAN EKSTRAKSI HISTOGRAM BERSAMAAN
# ==========================================
def extract_histogram_features(img_dir):
    X_features = []
    y_labels = []

    # Path utama
    base_path = Path(img_dir)

    # Looping setiap sub-folder yang mewakili class (misal: 'siang', 'malam' atau 'day', 'night')
    # Kita abaikan folder test_dir dulu karena Tugas 2 meminta split 80:20 dari keseluruhan data.
    # Asumsinya gambar siang/malam ada di dalam folder 'images/training/' atau subfoldernya
    for folder in base_path.glob('*'):
        if folder.is_dir():
            label = folder.name

            # Looping setiap file gambar di dalam folder kelas tersebut
            for file_path in folder.glob('*.jpg'):
                try:
                    img = mpimg.imread(str(file_path))
                    if img is not None:
                        hist_features = []

                        # Cek apakah gambar berwarna (3 channel) atau grayscale
                        if len(img.shape) == 3:
                            for i in range(3): # Channel R, G, B
                                hist, _ = np.histogram(img[:, :, i], bins=32, range=(0, 256))
                                hist_features.extend(hist)
                        else:
                            hist, _ = np.histogram(img, bins=32, range=(0, 256))
                            hist_features.extend(hist)

                        X_features.append(hist_features)
                        y_labels.append(label)
                except Exception as e:
                    print(f"Gagal membaca {file_path}: {e}")

    return np.array(X_features), np.array(y_labels)


# Tentukan path sesuai struktur foldermu
# Berdasarkan screenshot, kamu menset train_dir = "images/training/"
train_dir = "images/training/"

print(f"Membaca gambar dan mengekstrak histogram dari: {train_dir}")
X_hist, y_img = extract_histogram_features(train_dir)

print(f"Total gambar yang diproses: {len(y_img)}")
print(f"Bentuk array fitur (X_hist): {X_hist.shape}")

# Pastikan data tidak kosong
if len(y_img) == 0:
    print("\nERROR: Tidak ada gambar yang ditemukan. Pastikan:")
    print("1. Path 'images/training/' benar.")
    print("2. Di dalam 'images/training/' terdapat sub-folder kelas (misal: folder 'siang' dan 'malam').")
    print("3. Gambar berformat .jpg.")
else:
    # ==========================================
    # 1. SPLIT DATA 80:20 (Sesuai instruksi Tugas 2)
    # ==========================================
    X_train, X_test, y_train, y_test = train_test_split(
        X_hist, y_img, test_size=0.2, random_state=42
    )

    print("\nData berhasil di-split (80:20).")
    print(f"Jumlah Data Latih: {len(y_train)}")
    print(f"Jumlah Data Uji: {len(y_test)}")

    # ==========================================
    # 2. MODELING (SVM Kernel RBF) & TUNING
    # ==========================================
    svm_model = SVC(kernel='rbf', random_state=42)

    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01, 0.001]
    }

    print("\nMemulai Hyperparameter Tuning GridSearchCV (CV=5)...")
    grid_search = GridSearchCV(svm_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_svm = grid_search.best_estimator_
    print(f"Hyperparameter Terbaik ditemukan: {grid_search.best_params_}")

    # ==========================================
    # 3. PREDIKSI DAN EVALUASI (Catat Performansi)
    # ==========================================
    y_pred = best_svm.predict(X_test)
    akurasi = accuracy_score(y_test, y_pred)

    print("\n==========================================")
    print("HASIL EVALUASI TUGAS 2")
    print("==========================================")
    print(f"Akurasi SVM (RBF) Siang/Malam: {akurasi:.4f} ({(akurasi*100):.2f}%)")
    print("\nLaporan Klasifikasi Detail:")
    print(classification_report(y_test, y_pred))

Membaca gambar dan mengekstrak histogram dari: images/training/
Total gambar yang diproses: 240
Bentuk array fitur (X_hist): (240, 96)

Data berhasil di-split (80:20).
Jumlah Data Latih: 192
Jumlah Data Uji: 48

Memulai Hyperparameter Tuning GridSearchCV (CV=5)...
Hyperparameter Terbaik ditemukan: {'C': 10, 'gamma': 'scale'}

HASIL EVALUASI TUGAS 2
Akurasi SVM (RBF) Siang/Malam: 1.0000 (100.00%)

Laporan Klasifikasi Detail:
              precision    recall  f1-score   support

         day       1.00      1.00      1.00        24
       night       1.00      1.00      1.00        24

    accuracy                           1.00        48
   macro avg       1.00      1.00      1.00        48
weighted avg       1.00      1.00      1.00        48

